<a href="https://colab.research.google.com/github/motunrayoade333/Motunrayo-s-portfolio/blob/main/Week_3_Data_Structures_%26_the_Supply_Chain_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import csv

filename = "supply_chain_dataset.csv"

with open(filename, newline="", encoding="utf-8") as f:
    orders = list(csv.DictReader(f))

print("Rows:", len(orders))
print("Columns:", list(orders[0].keys()))
print("First row:", orders[0])

Rows: 3000
Columns: ['order_id', 'order_date', 'product_category', 'quantity', 'unit_price', 'total_cost', 'supplier_name', 'supplier_location', 'lead_time_days', 'warehouse_destination', 'delivery_date', 'actual_delivery_days', 'delivery_status', 'on_time', 'product_defects', 'return_status']
First row: {'order_id': 'ORD-0001', 'order_date': '11/28/2022', 'product_category': 'Fasteners', 'quantity': '94', 'unit_price': '64.49', 'total_cost': '6062.06', 'supplier_name': 'SupplierE', 'supplier_location': 'Mexico', 'lead_time_days': '32', 'warehouse_destination': 'Warehouse_3', 'delivery_date': '2022-12-30', 'actual_delivery_days': '32', 'delivery_status': 'Delayed', 'on_time': 'Yes', 'product_defects': '0.0', 'return_status': 'No Return'}


In [4]:
# Task 1: Organise orders in a list

# Build a list of every order's total_cost as numbers
costs = [float(row["total_cost"]) for row in orders]

# Total spend and average order value
total_spend = sum(costs)
average_order = total_spend / len(costs)
print(f"Orders: {len(costs)}")
print(f"Total spend: {total_spend:,.2f}")
print(f"Average order value: {average_order:,.2f}")

# Sort a copy (sort() works in place and returns None)
sorted_costs = sorted(costs)

# Median: middle value, or the average of the two middle values
n = len(sorted_costs)
if n % 2 == 1:
    median = sorted_costs[n // 2]
else:
    median = (sorted_costs[n // 2 - 1] + sorted_costs[n // 2]) / 2
print(f"Median order value: {median:,.2f}")

# Five smallest and five largest
print("Five smallest:", sorted_costs[:5])
print("Five largest:", sorted_costs[-5:])

Orders: 3000
Total spend: 170,038,100.28
Average order value: 56,679.37
Median order value: 30,911.19
Five smallest: [-207.32462996732556, 335.26682978169913, 340.35, 340.52083157446225, 352.6]
Five largest: [765820.16, 807800.07, 991898.31, 1000681.24, 1720595.4]


In [6]:
def clean_supplier(name):
    """'Supplier A', 'suppliera', 'SupplierA' -> 'SupplierA'"""
    compact = name.strip().lower().replace(" ", "")
    return "Supplier" + compact[-1].upper()

def clean_category(category):
    """' Motors ' -> 'Motors'"""
    return category.strip()

In [7]:
# Task 2: Map categories to suppliers

category_suppliers = {}   # {category: {set of suppliers}}

for row in orders:
    category = clean_category(row["product_category"])
    supplier = clean_supplier(row["supplier_name"])

    # First time we see this category, start it with an empty set
    if category not in category_suppliers:
        category_suppliers[category] = set()

    category_suppliers[category].add(supplier)

# Print each category with its suppliers
for category in sorted(category_suppliers):
    print(f"{category}: {sorted(category_suppliers[category])}")

# Look up who supplies Pumps
print("\nPumps suppliers:", category_suppliers["Pumps"])

# Count unique suppliers across all categories
all_suppliers = set()
for suppliers in category_suppliers.values():
    all_suppliers |= suppliers      # |= adds every item from the other set
print("Unique suppliers:", len(all_suppliers))

Connectors: ['SupplierE']
Control Systems: ['SupplierA']
Fasteners: ['SupplierE']
Hydraulics: ['SupplierD']
Motors: ['SupplierA', 'SupplierB']
Pumps: ['SupplierC', 'SupplierD']
Sensors: ['SupplierB', 'SupplierC']
Valves: ['SupplierB', 'SupplierC']

Pumps suppliers: {'SupplierD', 'SupplierC'}
Unique suppliers: 5


In [8]:
# Task 3: Find unique values with sets

# Raw sets (before cleaning)
raw_suppliers = {row["supplier_name"] for row in orders}
raw_categories = {row["product_category"] for row in orders}
statuses = {row["delivery_status"] for row in orders}

# Cleaned sets (after using the helper functions)
clean_suppliers = {clean_supplier(row["supplier_name"]) for row in orders}
clean_categories = {clean_category(row["product_category"]) for row in orders}

print(f"Suppliers:  {len(raw_suppliers)} before, {len(clean_suppliers)} after")
print(f"Categories: {len(raw_categories)} before, {len(clean_categories)} after")
print(f"Statuses:   {len(statuses)} unique values")
print("Statuses:", sorted(statuses))

# Set operations on the Task 2 dictionary
motors = category_suppliers["Motors"]
pumps = category_suppliers["Pumps"]

print("\nSupply both Motors and Pumps:", motors & pumps)
print("Supply Motors or Pumps:", sorted(motors | pumps))
print("Supply Motors but not Pumps:", sorted(motors - pumps))

Suppliers:  15 before, 5 after
Categories: 8 before, 8 after
Statuses:   6 unique values
Statuses: ['Cancelled', 'Delayed', 'Delievered', 'Delivered', 'In-Transit', 'Pendng']

Supply both Motors and Pumps: set()
Supply Motors or Pumps: ['SupplierA', 'SupplierB', 'SupplierC', 'SupplierD']
Supply Motors but not Pumps: ['SupplierA', 'SupplierB']


In [9]:
# Task 4: Single-source risk report

print("SINGLE-SOURCE RISK REPORT")
print("-" * 40)

single_source_count = 0

for category in sorted(category_suppliers):
    suppliers = category_suppliers[category]
    if len(suppliers) == 1:
        single_source_count += 1
        only_supplier = list(suppliers)[0]   # pull the one item out of the set
        print(f"{category}: depends only on {only_supplier}")

print("-" * 40)
print(f"{single_source_count} of {len(category_suppliers)} categories are single-source")

SINGLE-SOURCE RISK REPORT
----------------------------------------
Connectors: depends only on SupplierE
Control Systems: depends only on SupplierA
Fasteners: depends only on SupplierE
Hydraulics: depends only on SupplierD
----------------------------------------
4 of 8 categories are single-source


In [10]:
# Task 5: Build the supplier network

network = {}

for row in orders:
    supplier = clean_supplier(row["supplier_name"])
    category = clean_category(row["product_category"])
    cost = float(row["total_cost"])

    # First time we see this supplier, create its entry
    if supplier not in network:
        network[supplier] = {
            "categories": set(),
            "order_count": 0,
            "total_spend": 0.0,
        }

    network[supplier]["categories"].add(category)
    network[supplier]["order_count"] += 1
    network[supplier]["total_spend"] += cost

# Query 1: everything about SupplierA
print("Everything about SupplierA:")
print(network["SupplierA"])

# Query 2: one line per supplier
print("\nSupplier network summary:")
for supplier in sorted(network):
    info = network[supplier]
    print(f"{supplier}: {info['order_count']} orders, "
          f"{len(info['categories'])} categories, "
          f"total spend {info['total_spend']:,.2f}")

Everything about SupplierA:
{'categories': {'Motors', 'Control Systems'}, 'order_count': 769, 'total_spend': 70637594.4608911}

Supplier network summary:
SupplierA: 769 orders, 2 categories, total spend 70,637,594.46
SupplierB: 659 orders, 3 categories, total spend 43,099,108.00
SupplierC: 492 orders, 3 categories, total spend 23,085,480.95
SupplierD: 433 orders, 2 categories, total spend 28,449,003.78
SupplierE: 647 orders, 2 categories, total spend 4,766,913.09


In [11]:
# Challenge 1: orders above 100,000
big_orders = [cost for cost in costs if cost > 100000]
percentage = len(big_orders) / len(costs) * 100
print(f"Orders above 100,000: {len(big_orders)}")
print(f"Share of all orders: {percentage:.2f}%")

# Challenge 2: supplier with the most categories
top_supplier = max(network, key=lambda s: len(network[s]["categories"]))
print(f"\nMost categories: {top_supplier} "
      f"({len(network[top_supplier]['categories'])} categories)")

# Challenge 3: group order costs by delivery status
costs_by_status = {}
for row in orders:
    status = row["delivery_status"]
    if status not in costs_by_status:
        costs_by_status[status] = []
    costs_by_status[status].append(float(row["total_cost"]))

delivered = costs_by_status["Delivered"]
print(f"\nAverage 'Delivered' order: {sum(delivered) / len(delivered):,.2f}")

Orders above 100,000: 478
Share of all orders: 15.93%

Most categories: SupplierC (3 categories)

Average 'Delivered' order: 57,473.02


In [13]:
"""After cleaning the supplier names with the helper function, the data shows 5 real suppliers
(SupplierA to SupplierE) and 8 product categories. Four categories are single-source: Connectors and
Fasteners depend only on SupplierE, Control Systems depends only on SupplierA, and Hydraulics depends
only on SupplierD. This is a risk because if one of those suppliers fails, the company has no other
way to get that product, and a failure at SupplierE would cut off two categories at once. SupplierA
receives the most spend, about 70.6 million out of a total of 170.0 million.

'After cleaning the supplier names with the helper function, the data shows 5 real suppliers (SupplierA to SupplierE) and 8 product categories. Four categories are single-source: Connectors and Fasteners depend only on SupplierE, Control Systems depends only on SupplierA, and Hydraulics depends only on SupplierD. This is a risk because if one of those suppliers fails, the company has no other way to get that product, and a failure at SupplierE would cut off two categories at once. SupplierA receives the most spend, about 70.6 million out of a total of 170.0 million.'